In [ ]:
# 🤖 Milestone 1: Edge Optimization for Robot Deployment - CLEAN VERSION

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import os
import sys
import time

# Add project path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.utils.preprocessing import load_fast_csv, window_data

# Set seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print("🤖 Edge Optimization for Robot Deployment - Clean Implementation")
print("="*70)

# ============================================================================
# 1. DATA LOADING & PREPROCESSING
# ============================================================================

def load_optimization_dataset(data_folder, n_files=3, window_size=512, samples_per_file=200):
    """Load and prepare dataset for model optimization"""
    
    files = [f for f in os.listdir(data_folder) if f.endswith("_fast.csv")][:n_files]
    print(f"Loading {len(files)} files for optimization...")
    
    all_windows, all_rul = [], []
    
    for i, fname in enumerate(files):
        signal = load_fast_csv(os.path.join(data_folder, fname))
        windows = window_data(signal, window_size=window_size)[:samples_per_file]
        
        # Generate realistic RUL labels
        max_rul = len(windows)
        rul = np.arange(max_rul, 0, -1).astype(float)
        
        all_windows.extend(windows)
        all_rul.extend(rul)
    
    X = np.array(all_windows)
    y = np.array(all_rul)
    
    # Normalize features
    X_reshaped = X.reshape(-1, 2)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_reshaped)
    X = X_scaled.reshape(X.shape)
    
    print(f"Dataset prepared: {X.shape[0]} samples, RUL range: {y.min():.1f}-{y.max():.1f}")
    return X, y, scaler

# ============================================================================
# 2. BASELINE MODEL
# ============================================================================

def create_baseline_model(input_shape):
    """Create baseline model for comparison"""
    return models.Sequential([
        layers.Conv1D(64, 5, activation='relu', input_shape=input_shape),
        layers.MaxPooling1D(2),
        layers.Conv1D(32, 3, activation='relu'),
        layers.MaxPooling1D(2),
        layers.LSTM(50, return_sequences=True),
        layers.LSTM(25),
        layers.Dense(50, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(25, activation='relu'),
        layers.Dense(1)
    ])

# ============================================================================
# 3. OPTIMIZED ARCHITECTURES
# ============================================================================

def create_lightweight_model(input_shape):
    """Create lightweight model with separable convolutions"""
    return models.Sequential([
        layers.SeparableConv1D(32, 5, activation='relu', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        
        layers.SeparableConv1D(16, 3, activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        
        layers.LSTM(24, return_sequences=False),
        layers.BatchNormalization(),
        
        layers.Dense(16, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(1)
    ])

def create_edge_optimized_model(input_shape):
    """Create ultra-lightweight model for edge deployment"""
    return models.Sequential([
        layers.SeparableConv1D(12, 5, activation='relu', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        
        layers.SeparableConv1D(8, 3, activation='relu'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling1D(),  # Replace LSTM for TFLite compatibility
        
        layers.Dense(8, activation='relu'),
        layers.Dropout(0.1),
        layers.Dense(1)
    ])

# ============================================================================
# 4. QUANTIZATION UTILITIES
# ============================================================================

def quantize_model(model, X_train_sample):
    """Convert model to quantized TensorFlow Lite format"""
    
    def representative_dataset():
        for i in range(100):
            idx = np.random.randint(0, len(X_train_sample))
            yield [X_train_sample[idx:idx+1].astype(np.float32)]
    
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_types = [tf.int8]
    converter.inference_input_type = tf.float32
    converter.inference_output_type = tf.float32
    
    return converter.convert()

# ============================================================================
# 5. PERFORMANCE MEASUREMENT
# ============================================================================

def measure_inference_time(model, X_test, n_iterations=100):
    """Measure average inference time"""
    start_time = time.time()
    for _ in range(n_iterations):
        _ = model.predict(X_test[:1], verbose=0)
    return (time.time() - start_time) / n_iterations * 1000

def evaluate_model(model, X_test, y_test, model_name):
    """Evaluate model performance and efficiency"""
    pred = model.predict(X_test, verbose=0)
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    inference_time = measure_inference_time(model, X_test)
    
    return {
        'name': model_name,
        'mae': mae,
        'r2': r2,
        'parameters': model.count_params(),
        'inference_time': inference_time,
        'predictions': pred
    }

# ============================================================================
# 6. KNOWLEDGE DISTILLATION
# ============================================================================

class KnowledgeDistillationModel(keras.Model):
    """Knowledge distillation for model compression"""
    
    def __init__(self, teacher_model, student_model, alpha=0.7, temperature=3):
        super().__init__()
        self.teacher = teacher_model
        self.student = student_model
        self.alpha = alpha
        self.temperature = temperature

    def call(self, inputs, training=False):
        return self.student(inputs, training=training)

    def train_step(self, data):
        x, y_true = data
        teacher_pred = self.teacher(x, training=False)
        
        with tf.GradientTape() as tape:
            student_pred = self.student(x, training=True)
            
            # Distillation loss
            soft_teacher = teacher_pred / self.temperature
            soft_student = student_pred / self.temperature
            distillation_loss = tf.reduce_mean(
                tf.square(soft_teacher - soft_student)
            ) * (self.temperature ** 2)
            
            # Student loss
            student_loss = tf.reduce_mean(tf.square(y_true - student_pred))
            
            # Combined loss
            total_loss = self.alpha * distillation_loss + (1 - self.alpha) * student_loss

        gradients = tape.gradient(total_loss, self.student.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.student.trainable_variables))
        
        return {"loss": total_loss, "student_loss": student_loss, "distillation_loss": distillation_loss}

def create_student_model(input_shape):
    """Create compact student model for knowledge distillation"""
    return models.Sequential([
        layers.Conv1D(16, 5, activation='relu', input_shape=input_shape),
        layers.MaxPooling1D(2),
        layers.Conv1D(8, 3, activation='relu'),
        layers.GlobalAveragePooling1D(),
        layers.Dense(12, activation='relu'),
        layers.Dropout(0.1),
        layers.Dense(1)
    ])

# ============================================================================
# 7. MAIN OPTIMIZATION PIPELINE
# ============================================================================

def run_edge_optimization(data_folder="../data/SMARTPDM_dataset.csv"):
    """Main pipeline for edge optimization"""
    
    # 1. Load and prepare data
    X, y, scaler = load_optimization_dataset(data_folder)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    print(f"Data splits: Train={X_train.shape[0]}, Test={X_test.shape[0]}")
    
    results = {}
    
    # 2. Train baseline model
    print("\n📊 Training baseline model...")
    baseline_model = create_baseline_model(X_train.shape[1:])
    baseline_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    baseline_model.fit(X_train, y_train, epochs=20, batch_size=32, 
                      validation_data=(X_test, y_test), verbose=0,
                      callbacks=[EarlyStopping(patience=5, restore_best_weights=True)])
    
    results['baseline'] = evaluate_model(baseline_model, X_test, y_test, 'Baseline')
    
    # 3. Train lightweight model
    print("⚡ Training lightweight model...")
    lightweight_model = create_lightweight_model(X_train.shape[1:])
    lightweight_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    lightweight_model.fit(X_train, y_train, epochs=25, batch_size=32,
                         validation_data=(X_test, y_test), verbose=0,
                         callbacks=[EarlyStopping(patience=7, restore_best_weights=True)])
    
    results['lightweight'] = evaluate_model(lightweight_model, X_test, y_test, 'Lightweight')
    
    # 4. Train edge-optimized model
    print("🚀 Training edge-optimized model...")
    edge_model = create_edge_optimized_model(X_train.shape[1:])
    edge_model.compile(optimizer=optimizers.Adam(learning_rate=0.002), loss='mse', metrics=['mae'])
    edge_model.fit(X_train, y_train, epochs=30, batch_size=32,
                  validation_data=(X_test, y_test), verbose=0,
                  callbacks=[
                      EarlyStopping(patience=8, restore_best_weights=True),
                      ReduceLROnPlateau(patience=5, factor=0.5, min_lr=1e-6)
                  ])
    
    results['edge'] = evaluate_model(edge_model, X_test, y_test, 'Edge-Optimized')
    
    # 5. Knowledge distillation
    print("🧠 Training with knowledge distillation...")
    student_model = create_student_model(X_train.shape[1:])
    kd_model = KnowledgeDistillationModel(lightweight_model, student_model)
    kd_model.compile(optimizer='adam')
    kd_model.fit(X_train, y_train, epochs=15, batch_size=32,
                 validation_data=(X_test, y_test), verbose=0)
    
    results['student'] = evaluate_model(student_model, X_test, y_test, 'Student (KD)')
    
    # 6. Create quantized models
    print("🔧 Creating quantized models...")
    edge_quantized = quantize_model(edge_model, X_train[:1000])
    
    # Save models
    edge_model.save("temp_edge_model.h5")
    edge_size = os.path.getsize("temp_edge_model.h5")
    
    with open("edge_optimized_quantized.tflite", 'wb') as f:
        f.write(edge_quantized)
    quantized_size = os.path.getsize("edge_optimized_quantized.tflite")
    
    # 7. Results summary
    print("\n📊 OPTIMIZATION RESULTS:")
    print("-" * 80)
    print(f"{'Model':<15} {'Parameters':<12} {'MAE':<8} {'R²':<8} {'Time(ms)':<10}")
    print("-" * 80)
    
    for result in results.values():
        print(f"{result['name']:<15} {result['parameters']:<12,} {result['mae']:<8.2f} "
              f"{result['r2']:<8.3f} {result['inference_time']:<10.1f}")
    
    print(f"\n📱 Model Sizes:")
    print(f"Edge Model (H5): {edge_size/1024:.1f} KB")
    print(f"Quantized (TFLite): {quantized_size/1024:.1f} KB")
    print(f"Size Reduction: {(1 - quantized_size/edge_size)*100:.1f}%")
    
    # 8. Deployment readiness check
    edge_result = results['edge']
    baseline_result = results['baseline']
    
    size_check = quantized_size/1024/1024 < 10  # <10MB
    speed_check = edge_result['inference_time'] < 50  # <50ms
    param_check = edge_result['parameters'] < 50000  # <50K params
    performance_check = edge_result['mae'] < baseline_result['mae'] * 1.2  # <20% degradation
    
    print(f"\n🎯 Deployment Readiness:")
    print(f"Model Size (<10MB): {'✅' if size_check else '❌'} {quantized_size/1024/1024:.2f}MB")
    print(f"Inference Speed (<50ms): {'✅' if speed_check else '❌'} {edge_result['inference_time']:.1f}ms")
    print(f"Parameters (<50K): {'✅' if param_check else '❌'} {edge_result['parameters']:,}")
    print(f"Performance (<20% loss): {'✅' if performance_check else '❌'} {(edge_result['mae']/baseline_result['mae']-1)*100:+.1f}%")
    
    deployment_ready = all([size_check, speed_check, param_check, performance_check])
    print(f"\n🚀 MILESTONE 1 STATUS: {'✅ SUCCESS' if deployment_ready else '⚠️ PARTIAL SUCCESS'}")
    
    return {
        'models': {
            'baseline': baseline_model,
            'lightweight': lightweight_model,
            'edge': edge_model,
            'student': student_model
        },
        'results': results,
        'quantized_model': edge_quantized,
        'deployment_ready': deployment_ready,
        'scaler': scaler
    }

# ============================================================================
# 8. EXECUTE IF RUN DIRECTLY
# ============================================================================

if __name__ == "__main__":
    optimization_results = run_edge_optimization()
    print("\n✅ Edge optimization pipeline complete!")